In [1]:
import gc
import json
import os
import time
import uuid
from dataclasses import dataclass, field, asdict
from typing import Optional

from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, set_seed

import torch
from txtai import Embeddings
import pandas as pd
import numpy as np
#from datasets import load_dataset
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
import bm25s
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize
import nltk
from FlagEmbedding import FlagReranker
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from typing import List, Dict


/home/manavka/anaconda3/envs/RAG/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.15.0
True
NVIDIA RTX 2000 Ada Generation Laptop GPU


<h1>Define Functions</h1>

In [3]:
# Load ST model once for efficiency
#_st_model = SentenceTransformer("all-MiniLM-L6-v2")


def get_dense_embeddings(context_ids, context_docs):
    documents = list(zip(context_ids, context_docs))

    dense_embeddings = Embeddings(path="neuml/pubmedbert-base-embeddings", content=True)
    dense_embeddings.index(documents)
    return dense_embeddings

def get_sparse_embeddings(context_ids, context_docs):
    contexts_train_tokens = bm25s.tokenize(context_docs, stopwords="en")
    sparse_embeddings = bm25s.BM25(corpus=np.array(context_ids))    
    sparse_embeddings.index(contexts_train_tokens)
    return sparse_embeddings

def reciprocal_rank_fusion(ranks: List[List[str]], k: int = 60) -> List[str]:
    """
    ranks: list of lists of document IDs from each retrieval method
    k: constant for rank discount
    """
    doc_scores = {doc: 0.0 for doc in set().union(*ranks)}
    for rank_list in ranks:
        for rank, doc in enumerate(rank_list, start=1):
            doc_scores[doc] += 1 / (rank + k)
    return sorted(doc_scores.keys(), key=lambda d: doc_scores[d], reverse=True)

# Example usage:
# list1 = ["docA", "docB", "docC"]
# list2 = ["docB", "docC", "docD"]
# final = reciprocal_rank_fusion([list1, list2], k=60)
# print(final)

def retrieve_and_fuse(query,dense_embeddings, sparse_embeddings,corpus, dense_limit=25, sparse_limit=25, rrf_rank_discount=60):
    dense_list= dense_embeddings.search(query, limit=dense_limit)
    dense_ids = [dict['id'] for dict in dense_list]
    query_tokens = bm25s.tokenize(query, stopwords="en")
    sparse_ids, _ = sparse_embeddings.retrieve(query_tokens, k=sparse_limit)
    sparse_ids = sparse_ids[0].tolist()
    fused_ids = reciprocal_rank_fusion([dense_ids, sparse_ids], k=rrf_rank_discount)
    fused_results = [
    {
        "doc_id": int(doc_id),
        "text": corpus[int(doc_id)]
    }
    for doc_id in fused_ids
]

    return fused_results


def rerank_documents(query, fused_results, reranker):
    """
    Rerank documents using BGE cross encoder.

    Parameters
    ----------
    query : str
    fused_results : list[dict]
        Must contain keys:
            - doc_id
            - text

    Returns
    -------
    list[dict]
        Sorted by reranker score
    """

    query_doc_pairs = [
        [query, doc["text"]]
        for doc in fused_results
    ]

    scores = reranker.compute_score(
        query_doc_pairs,
        normalize=True
    )

    for doc, score in zip(fused_results, scores):
        doc["rerank_score"] = float(score)

    reranked = sorted(
        fused_results,
        key=lambda x: x["rerank_score"],
        reverse=True,
    )

    return reranked
def initialize_model_and_tokenizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Configure BitsAndBytes for 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    return model, tokenizer

def structure_message(query, top_docs):
    """
    Structures a message for the model to answer a biomedical question using the provided context.

    Args:
        query (str): The biomedical question.
        top_docs (list[str]): The top documents providing context. The list is expected to be ordered by relevance and 

    Returns:
        list[dict]: A list of messages structured for the model.
    """
    context = "\n\n".join(top_docs)
    messages = [
    {
        "role": "system",
        "content": (
            "You are a biomedical question answering assistant. "
            "Answer using only the provided context."
        )
    },
    {
    "role": "user",
    "content": f"""
        Question:
        {query}

        Context:
        {context}
        """
    }
    ]
    return messages

def serve_model(model, tokenizer, query, top_docs):
    message = structure_message(query, top_docs)
    text = tokenizer.apply_chat_template(message,tokenize=False,add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=150)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

def evaluate(reference_answer: str, candidate_answer: str, _st_model: SentenceTransformer) -> dict:
    """
    Computes BLEU, ROUGE, METEOR, and Semantic Text Similarity.

    Args:
        reference_answer (str): Ground-truth answer.
        candidate_answer (str): Model-generated answer.
        _st_model (SentenceTransformer): Preloaded SentenceTransformer model for semantic similarity.

    Returns:
        dict: Evaluation metrics.
    """

    # --------------------
    # BLEU
    # --------------------
    reference_tokens = [word_tokenize(reference_answer.lower())]
    candidate_tokens = word_tokenize(candidate_answer.lower())

    bleu = sentence_bleu(
        reference_tokens,
        candidate_tokens,
        smoothing_function=SmoothingFunction().method1,
    )

    # --------------------
    # ROUGE
    # --------------------
    rouge = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True,
    )

    rouge_scores = rouge.score(reference_answer, candidate_answer)

    # --------------------
    # METEOR
    # --------------------
    meteor = meteor_score(
        [word_tokenize(reference_answer)],
        word_tokenize(candidate_answer),
    )

    # --------------------
    # Semantic Similarity (ST)
    # --------------------
    ref_emb = _st_model.encode([reference_answer])
    cand_emb = _st_model.encode([candidate_answer])

    semantic_similarity = cosine_similarity(ref_emb, cand_emb)[0][0]

    return {
        "BLEU": round(bleu, 4),
        "ROUGE-1": round(rouge_scores["rouge1"].fmeasure, 4),
        "ROUGE-2": round(rouge_scores["rouge2"].fmeasure, 4),
        "ROUGE-L": round(rouge_scores["rougeL"].fmeasure, 4),
        "METEOR": round(meteor, 4),
        "ST_SIMILARITY": round(float(semantic_similarity), 4),
    }


<h1>LORA Functions</h1>

In [4]:


ATTN_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
ATTN_MLP_MODULES = ATTN_MODULES + ["gate_proj", "up_proj", "down_proj"]
METRIC_COLS = ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR", "ST_SIMILARITY"]


@dataclass
class LoraExperimentConfig:
    run_name: str = "lora_default"
    base_model: str = "Qwen/Qwen2.5-7B-Instruct"
    # LoRA
    r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: List[str] = field(default_factory=lambda: list(ATTN_MODULES))
    # Training
    learning_rate: float = 2e-4
    num_train_epochs: float = 1.0
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    warmup_ratio: float = 0.03
    lr_scheduler_type: str = "cosine"
    weight_decay: float = 0.0
    max_grad_norm: float = 0.3
    optim: str = "paged_adamw_8bit"
    # Sequence / loss
    max_seq_len: int = 1536
    max_answer_tokens: int = 256
    answer_only_loss: bool = True
    seed: int = 42
    output_dir: str = "lora_runs"


# ---------------------------------------------------------------------------
# Data: run the retrieval + rerank pipeline once and cache, so every ablation
# run (baseline and LoRA) sees identical context.
# ---------------------------------------------------------------------------
def build_rag_records(df, dense_embeddings, sparse_embeddings, corpus, reranker,
                      dense_limit=25, sparse_limit=25, rank_threshold=0.5,
                      max_docs=3, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        with open(cache_path) as f:
            return [json.loads(line) for line in f]

    records = []
    for qid, row in df.iterrows():
        fused = retrieve_and_fuse(row["QUESTION"], dense_embeddings, sparse_embeddings, corpus,
                                  dense_limit=dense_limit, sparse_limit=sparse_limit)
        reranked = rerank_documents(row["QUESTION"], fused, reranker)
        kept = [d for d in reranked if d["rerank_score"] > rank_threshold][:max_docs]
        records.append({
            "qid": int(qid),
            "question": row["QUESTION"],
            "answer": row["LONG_ANSWER"],
            "docs": [d["text"] for d in kept],
            "doc_ids": [d["doc_id"] for d in kept],
            "rerank_scores": [d["rerank_score"] for d in kept],
            "gold_doc_retrieved": int(qid) in [d["doc_id"] for d in kept],
        })

    if cache_path:
        with open(cache_path, "w") as f:
            for r in records:
                f.write(json.dumps(r) + "\n")
    return records


# ---------------------------------------------------------------------------
# Prompt building / tokenization (shared by training and inference)
# ---------------------------------------------------------------------------
def build_prompt_text(tokenizer, question, docs, max_prompt_tokens):
    """Chat-formatted prompt with the context truncated to fit max_prompt_tokens."""
    empty = tokenizer.apply_chat_template(structure_message(question, []), tokenize=False, add_generation_prompt=True)
    overhead = len(tokenizer(empty, add_special_tokens=False)["input_ids"])
    budget = max(max_prompt_tokens - overhead, 0)

    ctx_ids = tokenizer("\n\n".join(docs), add_special_tokens=False)["input_ids"][:budget]
    context = tokenizer.decode(ctx_ids)
    return tokenizer.apply_chat_template(structure_message(question, [context]), tokenize=False, add_generation_prompt=True)


def tokenize_example(tokenizer, record, max_seq_len, max_answer_tokens, answer_only_loss):
    # The assistant suffix (answer + end-of-turn tokens) doesn't depend on context, so derive it once from an empty prompt.
    msgs = structure_message(record["question"], [])
    prompt_only = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(msgs + [{"role": "assistant", "content": record["answer"]}], tokenize=False)
    answer_ids = tokenizer(full[len(prompt_only):], add_special_tokens=False)["input_ids"][:max_answer_tokens]

    prompt_text = build_prompt_text(tokenizer, record["question"], record["docs"], max_seq_len - len(answer_ids))
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

    input_ids = (prompt_ids + answer_ids)[:max_seq_len]
    if answer_only_loss:
        labels = ([-100] * len(prompt_ids) + answer_ids)[:max_seq_len]
    else:
        labels = list(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": [1] * len(input_ids)}


def make_collator(pad_token_id):
    def collate(features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "labels": [], "attention_mask": []}
        for f in features:
            pad = max_len - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [pad_token_id] * pad)
            batch["labels"].append(f["labels"] + [-100] * pad)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad)
        return {k: torch.tensor(v) for k, v in batch.items()}
    return collate


# ---------------------------------------------------------------------------
# Model loading (same 4-bit NF4 setup for baseline and LoRA => fair comparison)
# ---------------------------------------------------------------------------
def _compute_dtype():
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def load_quantized_base(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=_compute_dtype(),
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb_config, device_map="auto", torch_dtype=_compute_dtype()
    )
    return model, tokenizer


def load_decoder(model_name, adapter_path=None):
    """Load the base decoder, optionally with a trained LoRA adapter on top."""
    model, tokenizer = load_quantized_base(model_name)
    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    return model, tokenizer


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def train_lora(cfg: LoraExperimentConfig, train_records, val_records=None):
    """QLoRA fine-tune. Returns (model, tokenizer, training_info)."""
    set_seed(cfg.seed)
    run_dir = os.path.join(cfg.output_dir, cfg.run_name)
    os.makedirs(run_dir, exist_ok=True)

    model, tokenizer = load_quantized_base(cfg.base_model)
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(
        r=cfg.r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=cfg.target_modules,
        bias="none",
        task_type="CAUSAL_LM",
    ))
    trainable_params, total_params = model.get_nb_trainable_parameters()

    tok = lambda r: tokenize_example(tokenizer, r, cfg.max_seq_len, cfg.max_answer_tokens, cfg.answer_only_loss)
    train_ds = [tok(r) for r in train_records]
    val_ds = [tok(r) for r in val_records] if val_records else None

    bf16 = _compute_dtype() == torch.bfloat16
    args = TrainingArguments(
        output_dir=run_dir,
        num_train_epochs=cfg.num_train_epochs,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        lr_scheduler_type=cfg.lr_scheduler_type,
        warmup_ratio=cfg.warmup_ratio,
        weight_decay=cfg.weight_decay,
        max_grad_norm=cfg.max_grad_norm,
        optim=cfg.optim,
        bf16=bf16,
        fp16=not bf16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=10,
        eval_strategy="epoch" if val_ds else "no",  # `evaluation_strategy` on transformers < 4.41
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        seed=cfg.seed,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=make_collator(tokenizer.pad_token_id),
    )

    start = time.time()
    train_result = trainer.train()
    train_time = time.time() - start

    adapter_path = os.path.join(run_dir, "adapter")
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    with open(os.path.join(run_dir, "lora_config.json"), "w") as f:
        json.dump(asdict(cfg), f, indent=2)

    final_eval_loss = None
    if val_ds:
        final_eval_loss = trainer.evaluate().get("eval_loss")

    training_info = {
        "adapter_path": adapter_path,
        "train_loss": train_result.metrics.get("train_loss"),
        "final_eval_loss": final_eval_loss,
        "train_runtime_sec": round(train_time, 1),
        "trainable_params": int(trainable_params),
        "total_params": int(total_params),
        "trainable_pct": round(100 * trainable_params / total_params, 4),
        "n_train": len(train_ds),
        "n_val": len(val_ds) if val_ds else 0,
        "compute_dtype": "bf16" if bf16 else "fp16",
        "quantization": "4bit-nf4-double-quant",
        "loss_history": [h for h in trainer.state.log_history if "loss" in h or "eval_loss" in h],
    }

    model.config.use_cache = True
    model.eval()
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    return model, tokenizer, training_info


# ---------------------------------------------------------------------------
# Inference + evaluation
# ---------------------------------------------------------------------------
@torch.no_grad()
def generate_answer(model, tokenizer, record, max_prompt_tokens, max_new_tokens=256):
    prompt_text = build_prompt_text(tokenizer, record["question"], record["docs"], max_prompt_tokens)
    inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def evaluate_decoder(model, tokenizer, records, _st_model, max_prompt_tokens=1280, max_new_tokens=256):
    """Returns (per_example_df, summary_metrics_dict)."""
    rows = []
    start = time.time()
    for i, rec in enumerate(records):
        candidate = generate_answer(model, tokenizer, rec, max_prompt_tokens, max_new_tokens)
        scores = evaluate(rec["answer"], candidate, _st_model)
        rows.append({
            "qid": rec["qid"],
            "question": rec["question"],
            "reference": rec["answer"],
            "candidate": candidate,
            "num_docs": len(rec["docs"]),
            "gold_doc_retrieved": rec.get("gold_doc_retrieved"),
            **scores,
        })
        if (i + 1) % 25 == 0:
            print(f"  evaluated {i + 1}/{len(records)}")

    per_example = pd.DataFrame(rows)
    summary = {m: round(float(per_example[m].mean()), 4) for m in METRIC_COLS}
    summary.update({f"{m}_std": round(float(per_example[m].std()), 4) for m in METRIC_COLS})
    summary["n_eval"] = len(per_example)
    summary["eval_runtime_sec"] = round(time.time() - start, 1)
    return per_example, summary


# ---------------------------------------------------------------------------
# Experiment logging
# ---------------------------------------------------------------------------
def log_experiment(log_path, run_name, summary, lora_params=None, training_info=None,
                   retrieval_params=None, eval_params=None, per_example=None, output_dir="lora_runs"):
    """Append one run to a JSONL log; per-example predictions go to <output_dir>/<run_name>/predictions.csv."""
    run_id = f"{run_name}_{uuid.uuid4().hex[:8]}"
    entry = {
        "run_id": run_id,
        "run_name": run_name,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "uses_lora": lora_params is not None,
        "metrics": summary,
        "lora_params": lora_params,
        "training_info": training_info,
        "retrieval_params": retrieval_params,
        "eval_params": eval_params,
    }
    os.makedirs(os.path.dirname(log_path) or ".", exist_ok=True)
    with open(log_path, "a") as f:
        f.write(json.dumps(entry, default=str) + "\n")

    if per_example is not None:
        run_dir = os.path.join(output_dir, run_name)
        os.makedirs(run_dir, exist_ok=True)
        per_example.to_csv(os.path.join(run_dir, "predictions.csv"), index=False)
    return run_id


def load_experiment_log(log_path):
    """Flattened DataFrame of all logged runs (loss_history dropped for readability)."""
    with open(log_path) as f:
        entries = [json.loads(line) for line in f]
    for e in entries:
        if e.get("training_info"):
            e["training_info"].pop("loss_history", None)
    return pd.json_normalize(entries)


# ---------------------------------------------------------------------------
# Ablation runner
# ---------------------------------------------------------------------------
def run_ablation(configs: List[LoraExperimentConfig], train_records, eval_records, _st_model,
                 log_path="lora_runs/ablation_log.jsonl", retrieval_params=None, val_records=None,
                 include_baseline=True, base_model="Qwen/Qwen2.5-7B-Instruct",
                 max_prompt_tokens=1280, max_new_tokens=256):
    eval_params = {"max_prompt_tokens": max_prompt_tokens, "max_new_tokens": max_new_tokens,
                   "decoding": "greedy", "n_eval": len(eval_records)}

    if include_baseline:
        print("=== baseline (no LoRA) ===")
        model, tokenizer = load_decoder(base_model)
        per_example, summary = evaluate_decoder(model, tokenizer, eval_records, _st_model, max_prompt_tokens, max_new_tokens)
        log_experiment(log_path, "baseline_no_lora", summary, retrieval_params=retrieval_params,
                       eval_params=eval_params, per_example=per_example,
                       training_info={"quantization": "4bit-nf4-double-quant"})
        print(summary)
        del model, tokenizer
        free_gpu()

    for cfg in configs:
        print(f"=== {cfg.run_name} ===")
        model, tokenizer, training_info = train_lora(cfg, train_records, val_records)
        per_example, summary = evaluate_decoder(model, tokenizer, eval_records, _st_model, max_prompt_tokens, max_new_tokens)
        log_experiment(log_path, cfg.run_name, summary, lora_params=asdict(cfg), training_info=training_info,
                       retrieval_params=retrieval_params, eval_params=eval_params,
                       per_example=per_example, output_dir=cfg.output_dir)
        print(summary)
        del model, tokenizer
        free_gpu()

    return load_experiment_log(log_path)


<h1>Prepare Data</h1>


In [5]:
source_data = pd.read_json('pubmedqa/data/pqaa_train_set.json').T
source_data['id'] = source_data.index.astype(int)
train = source_data.iloc[0:700]
test = source_data.iloc[700:1000]

In [7]:
# Initialize embeddings and reranker models

QUERY_ID = 16097998
QUERY = train.loc[QUERY_ID, 'QUESTION']
RANK_THRESHOLD=0.5 # ALL RERANKED DOCUMENTS WITH RERANK SCORE ABOVE THIS THRESHOLD WILL BE KEPT
DECODER_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct" # decoder model for generating response
CORPUS = {
    idx: " ".join(contexts)
    for idx, contexts in zip(
        train.index,
        train["CONTEXTS"]
    )
}
TRUE_RESPONSE = train.loc[QUERY_ID, 'LONG_ANSWER']

_st_model = SentenceTransformer("all-MiniLM-L6-v2") # _st_model for semantic similarity

#reranker is for ranking layer
reranker = FlagReranker(
    "BAAI/bge-reranker-v2-m3",
    use_fp16=True
)



contexts_docs_train = [" ".join(contexts) for contexts in train["CONTEXTS"]]
context_ids_train = train.index.astype(str).tolist()
#documents_train = list(zip(context_ids_train, contexts_docs_train))

contexts_docs_test = [" ".join(contexts) for contexts in test["CONTEXTS"]]
context_ids_test = test.index.astype(str).tolist()
#documents_test = list(zip(context_ids_test, contexts_docs_test))


print('train length:', len(train))
print('test length:', len(test))


dense_embeddings = get_dense_embeddings(context_ids_train, contexts_docs_train)
sparse_embeddings = get_sparse_embeddings(context_ids_train, contexts_docs_train)


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2569.53it/s]


train length: 700
test length: 300


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1464.56it/s]


<h1>Driver Function</h1>

In [8]:
def driver(query, reference_answer, dense_embeddings, sparse_embeddings, corpus, context_ids, context_docs,  _st_model=None, reranker=None,dense_limit=25, sparse_limit=25,
           generator_model=None, generator_tokenizer=None):

    if dense_embeddings is None:
        dense_embeddings = get_dense_embeddings(context_ids, context_docs)
    if sparse_embeddings is None:
        sparse_embeddings = get_sparse_embeddings(context_ids, context_docs)
    if _st_model is None:
        _st_model = SentenceTransformer("all-MiniLM-L6-v2")
    if reranker is None:
        reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)
        


    fused_results = retrieve_and_fuse(query, dense_embeddings, sparse_embeddings, corpus, dense_limit=dense_limit, sparse_limit=sparse_limit, rrf_rank_discount = 60)
    
    # Ranking layer
    
    top_k = rerank_documents(
        query=query,
        fused_results=fused_results,
        reranker=reranker
    )
    # filtration for lowly ranked documents
    top_k = [candidate_ref for candidate_ref in top_k if candidate_ref['rerank_score'] > RANK_THRESHOLD]
    #top_k_ids = [candidate_ref['doc_id'] for candidate_ref in top_k]
    top_k_text = [candidate_ref['text'] for candidate_ref in top_k]
    #top_k_rerank_scores = [candidate_ref['rerank_score'] for candidate_ref in top_k]


    # BEGIN GENERATION

    if generator_model is None or generator_tokenizer is None:
        generator_model, generator_tokenizer = initialize_model_and_tokenizer(DECODER_MODEL_NAME)

    output = serve_model(generator_model, generator_tokenizer, query, top_k_text)

    candidate_response = output.split('\nassistant\n')[-1]

    score_dict = evaluate(reference_answer, candidate_response, _st_model)
    
    return score_dict, candidate_response, top_k

generator_model, generator_tokenizer = initialize_model_and_tokenizer(DECODER_MODEL_NAME)

score_dict, candidate_response, top_k = driver(
    query=QUERY,
    reference_answer=TRUE_RESPONSE,
    dense_embeddings=dense_embeddings,
    sparse_embeddings=sparse_embeddings,
    corpus=CORPUS,
    context_ids=context_ids_train,
    context_docs=contexts_docs_train,
    _st_model=_st_model,
    reranker=reranker,
    dense_limit=25,
    sparse_limit=25,
    generator_model=generator_model,
    generator_tokenizer=generator_tokenizer,
)

Loading weights: 100%|██████████| 339/339 [00:07<00:00, 46.14it/s]
/home/manavka/anaconda3/envs/RAG/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [9]:
print("QUESTION:\n", QUERY)
print('CANDIDATE RESPONSE:\n', candidate_response)
print('TRUE RESPONSE:\n', TRUE_RESPONSE)
print()
print(score_dict)

QUESTION:
 Is coeliac disease screening in risk groups justified?
CANDIDATE RESPONSE:
 Based on the information provided, coeliac disease screening in risk groups may be justified as it leads to high dietary compliance and similar outcomes in terms of quality of life, gastrointestinal symptoms, and bone mineral density compared to those detected through symptoms alone. However, the benefits of screening remain debatable, suggesting further research may be needed to fully justify widespread screening programs.
TRUE RESPONSE:
 Long-term dietary compliance in screen-detected patients was good. Quality of life and bone mineral density were comparable with those in non-coeliac subjects and the general population. Active screening in coeliac disease risk groups seems to be reasonable rather than harmful.

{'BLEU': 0.074, 'ROUGE-1': 0.4762, 'ROUGE-2': 0.1748, 'ROUGE-L': 0.2857, 'METEOR': 0.3957, 'ST_SIMILARITY': 0.9121}


<h1>LoRA Ablation</h1>

In [ ]:
RETRIEVAL_PARAMS = {"dense_model": "neuml/pubmedbert-base-embeddings", "sparse": "bm25s",
                    "reranker": "BAAI/bge-reranker-v2-m3", "dense_limit": 25, "sparse_limit": 25,
                    "rrf_k": 60, "rank_threshold": RANK_THRESHOLD, "max_docs": 3, "corpus": "train+test"}

# Combined train+test corpus so test questions can retrieve their own abstracts
combined = pd.concat([train, test])
CORPUS_ALL = {idx: " ".join(c) for idx, c in zip(combined.index, combined["CONTEXTS"])}
ids_all = combined.index.astype(str).tolist()
docs_all = [CORPUS_ALL[i] for i in combined.index]
dense_all = get_dense_embeddings(ids_all, docs_all)
sparse_all = get_sparse_embeddings(ids_all, docs_all)

rag_kwargs = dict(dense_limit=RETRIEVAL_PARAMS["dense_limit"], sparse_limit=RETRIEVAL_PARAMS["sparse_limit"],
                  rank_threshold=RETRIEVAL_PARAMS["rank_threshold"], max_docs=RETRIEVAL_PARAMS["max_docs"])
train_records_all = build_rag_records(train, dense_all, sparse_all, CORPUS_ALL, reranker,
                                      cache_path="lora_runs/rag_train.jsonl", **rag_kwargs)
test_records = build_rag_records(test, dense_all, sparse_all, CORPUS_ALL, reranker,
                                 cache_path="lora_runs/rag_test.jsonl", **rag_kwargs)

# 90/10 split of train for eval_loss tracking
n_val = len(train_records_all) // 10
val_records, train_records = train_records_all[:n_val], train_records_all[n_val:]

# Free retrieval models before loading the 7B decoder (<=12 GB VRAM)
del dense_all, sparse_all, dense_embeddings, sparse_embeddings, reranker
if "generator_model" in globals():
    del generator_model, generator_tokenizer
free_gpu()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2453.19it/s]
DEBUG:bm25s:Building index from IDs objects            


In [ ]:
# One-factor-at-a-time ablation around the default config
ablation_configs = [
    LoraExperimentConfig(run_name="r16_attn_default"),
    LoraExperimentConfig(run_name="r8_attn", r=8, lora_alpha=16),
    LoraExperimentConfig(run_name="r32_attn", r=32, lora_alpha=64),
    LoraExperimentConfig(run_name="r16_attn_mlp", target_modules=ATTN_MLP_MODULES),
    LoraExperimentConfig(run_name="r16_lr1e-4", learning_rate=1e-4),
    LoraExperimentConfig(run_name="r16_2epochs", num_train_epochs=2),
    LoraExperimentConfig(run_name="r16_dropout0.1", lora_dropout=0.1),
    LoraExperimentConfig(run_name="r16_fullseq_loss", answer_only_loss=False),
]

results_df = run_ablation(
    ablation_configs,
    train_records=train_records,
    eval_records=test_records,
    _st_model=_st_model,
    val_records=val_records,
    retrieval_params=RETRIEVAL_PARAMS,
    base_model=DECODER_MODEL_NAME,
    log_path="lora_runs/ablation_log.jsonl",
)

cols = ["run_name"] + [f"metrics.{m}" for m in METRIC_COLS] + [
    "lora_params.r", "lora_params.lora_alpha", "lora_params.target_modules", "lora_params.learning_rate",
    "lora_params.num_train_epochs", "lora_params.lora_dropout", "lora_params.answer_only_loss",
    "training_info.train_loss", "training_info.final_eval_loss", "training_info.trainable_pct"]
results_df[[c for c in cols if c in results_df.columns]]